# ET-SSL Evaluation & Export Notebook
Load the best checkpoint from notebook 01, run full evaluation on all datasets,
generate methodology discussion plots, and export production artifacts for the detection service.

## 1. Environment Setup

In [ ]:
import os, sys

REPO_URL   = "https://github.com/YOUR_USERNAME/Sentinel.git"   # ← update this
REPO_DIR   = "Sentinel"
HYBRID_DIR = f"{REPO_DIR}/hybrid-detection"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")

if HYBRID_DIR not in sys.path:
    sys.path.insert(0, HYBRID_DIR)

import subprocess
subprocess.run(["pip", "install", "-q", "optuna", "joblib", "tqdm"], check=True)
print("Repo ready.")

## 2. Imports

In [ ]:
import json, math
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                               recall_score, accuracy_score,
                               classification_report, roc_curve, confusion_matrix)

# ── Repo imports ──────────────────────────────────────────────────────────────
from config.constants import (
    FEATURE_DIM, FEATURE_NAMES,
    CONTINUOUS_INDICES, CONTINUOUS_MASK, CATEGORICAL_INDICES,
)
from config.column_maps import (
    CIC_DARKNET2020_COLUMN_MAP,
    CIC_IDS2018_COLUMN_MAP,
    CICFLOWMETER_PROTO_MAP,
    UNSW_MAP,
    LABEL_NORMAL,
)
from model.et_ssl import ETSSLModel
from feature_extractor.feature_builder import build_feature_vector, build_feature_matrix

print(f"Feature dim: {FEATURE_DIM}")
print("Imports OK.")

## 3. Configuration & Load Artifacts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_CHOICE = 'darknet'   # must match the model you trained in notebook 01
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_DIR = '/content/drive/MyDrive/Sentinel/datasets'
EXPORT_DIR  = '/content/drive/MyDrive/Sentinel/checkpoints'
PROD_DIR    = f'{EXPORT_DIR}/production'
PLOTS_DIR   = f'{EXPORT_DIR}/plots/{DATASET_CHOICE}'
os.makedirs(PROD_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Load meta from notebook 01
with open(f'{EXPORT_DIR}/meta_{DATASET_CHOICE}.json') as f:
    meta = json.load(f)

BEST_CONFIG = meta['best_config']
EMBED_DIM   = meta['embed_dim']
PROJ_DIM    = meta.get('proj_dim', 32)
THRESHOLD   = meta['threshold']

print(f"Dataset : {DATASET_CHOICE}")
print(f"Embed   : {EMBED_DIM}")
print(f"Threshold: {THRESHOLD:.6f}")
print(f"Val AUC : {meta['val_auc']:.4f}")
print(f"Test AUC: {meta['test_auc']:.4f}")
print(f"Config  : {json.dumps(BEST_CONFIG, indent=2)}")

## 4. Rebuild Model & Load Weights

In [ ]:
# ETSSLModel now accepts hidden_dims — reconstruct from best config
hidden_dims = tuple(BEST_CONFIG.get('hidden_dims', [128, 256, 128]))

model = ETSSLModel(
    hidden_dims=hidden_dims,
    embed_dim=EMBED_DIM,
    proj_dim=PROJ_DIM,
    dropout=BEST_CONFIG.get('dropout', 0.3),
).to(DEVICE)

state = torch.load(f'{EXPORT_DIR}/encoder_{DATASET_CHOICE}.pt', map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# Load scaler and centroid
scaler   = joblib.load(f'{EXPORT_DIR}/scaler_{DATASET_CHOICE}.joblib')
centroid = np.load(f'{EXPORT_DIR}/centroid_{DATASET_CHOICE}.npy')

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"Architecture: {hidden_dims} → {EMBED_DIM} → {PROJ_DIM}")
print(f"Centroid shape: {centroid.shape}")

## 5. Evaluate on All Datasets

In [ ]:
# Dataset configs — uses canonical column maps from repo
EVAL_DATASETS = {
    'darknet': {
        'path':       f'{DATASET_DIR}/darknet2020.csv',
        'col_map':    CIC_DARKNET2020_COLUMN_MAP,
        'label_normal': LABEL_NORMAL['darknet'],
        'fix_proto':  True,
    },
    'ids2018': {
        'path':       f'{DATASET_DIR}/cic_ids2018.csv',
        'col_map':    CIC_IDS2018_COLUMN_MAP,
        'label_normal': LABEL_NORMAL['ids2018'],
        'fix_proto':  True,
    },
    'unsw': {
        'path':       f'{DATASET_DIR}/unsw_nb15.csv',
        'col_map':    UNSW_MAP,
        'label_normal': LABEL_NORMAL['unsw'],
        'fix_proto':  False,
    },
}

@torch.no_grad()
def get_embeddings(model, X_np, device, bs=1024):
    model.eval()
    dl = DataLoader(torch.from_numpy(X_np).float(), batch_size=bs)
    return torch.cat([model.encode(b.to(device)) for b in dl]).cpu().numpy()

def load_and_evaluate(ds_name, ds_cfg, model, scaler, centroid, threshold, device):
    if not os.path.exists(ds_cfg['path']):
        print(f"  Skipping {ds_name}: file not found at {ds_cfg['path']}")
        return None

    df = pd.read_csv(ds_cfg['path'], low_memory=False)
    df = df.rename(columns=ds_cfg['col_map']).fillna(0)

    # Fix numeric proto for CICFlowMeter datasets
    if ds_cfg['fix_proto']:
        df['proto'] = df['proto'].map(CICFLOWMETER_PROTO_MAP).fillna('other')

    if len(df) > 200_000:
        df = df.sample(200_000, random_state=42)

    # Build features using canonical feature_builder
    records = df.to_dict(orient='records')
    X = build_feature_matrix(records)
    y = (df['label'] != ds_cfg['label_normal']).astype('int64').to_numpy()
    if len(y) != len(X):
        y = y[:len(X)]

    # Scale continuous columns only
    X_s = X.copy()
    X_s[:, CONTINUOUS_INDICES] = scaler.transform(
        X[:, CONTINUOUS_INDICES]
    ).astype(np.float32)

    z      = get_embeddings(model, X_s, device)
    scores = ((z - centroid[None, :]) ** 2).sum(axis=1)
    preds  = (scores > threshold).astype(int)

    auc = roc_auc_score(y, scores) if y.sum() > 0 and y.sum() < len(y) else float('nan')

    return {
        'y': y, 'scores': scores, 'preds': preds,
        'AUC':       round(auc, 4),
        'Accuracy':  round(accuracy_score(y, preds), 4),
        'Precision': round(precision_score(y, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y, preds, zero_division=0), 4),
        'F1':        round(f1_score(y, preds, zero_division=0), 4),
        'N':         len(y),
        'anomaly_pct': round(100 * y.mean(), 2),
    }

# Run evaluation
results = {}
for ds_name, ds_cfg in EVAL_DATASETS.items():
    print(f"\nEvaluating on {ds_name}...")
    result = load_and_evaluate(ds_name, ds_cfg, model, scaler, centroid, THRESHOLD, DEVICE)
    if result is not None:
        results[ds_name] = result
        print(classification_report(result['y'], result['preds'],
                                     target_names=['Normal', 'Anomaly']))

print("\n" + "=" * 60)
print("CROSS-DATASET EVALUATION SUMMARY")
print("=" * 60)
summary = {k: {m: v[m] for m in ['AUC','Accuracy','Precision','Recall','F1','N','anomaly_pct']}
           for k, v in results.items()}
print(pd.DataFrame(summary).T.to_string())

## 6. Evaluation Plots (Methodology Discussion)

In [ ]:
n_datasets = len(results)
if n_datasets == 0:
    print("No datasets evaluated — skipping plots.")
else:
    fig, axes = plt.subplots(2, n_datasets, figsize=(6 * n_datasets, 10))
    if n_datasets == 1:
        axes = axes.reshape(-1, 1)

    for col, (ds_name, res) in enumerate(results.items()):
        y, scores, preds = res['y'], res['scores'], res['preds']

        # Row 1 — ROC Curve
        ax = axes[0, col]
        fpr, tpr, _ = roc_curve(y, scores)
        ax.plot(fpr, tpr, color='darkorange', linewidth=2,
                label=f"AUC = {res['AUC']:.4f}")
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{ds_name} — ROC Curve', fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

        # Row 2 — Score distribution
        ax2 = axes[1, col]
        ax2.hist(scores[y == 0], bins=80, alpha=0.6, color='steelblue',
                 label='Normal', density=True)
        ax2.hist(scores[y == 1], bins=80, alpha=0.6, color='coral',
                 label='Anomaly', density=True)
        ax2.axvline(THRESHOLD, color='red', linestyle='--', linewidth=2,
                    label=f'Threshold = {THRESHOLD:.4f}')
        ax2.set_xlabel('Anomaly Score ||z - μ||²')
        ax2.set_ylabel('Density')
        ax2.set_title(f'{ds_name} — Score Distribution', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/cross_dataset_eval.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {PLOTS_DIR}/cross_dataset_eval.png")

## 7. Confusion Matrices

In [ ]:
import seaborn as sns

n = len(results)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, (ds_name, res) in zip(axes, results.items()):
        cm = confusion_matrix(res['y'], res['preds'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal', 'Anomaly'],
                    yticklabels=['Normal', 'Anomaly'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title(f'{ds_name} (F1={res["F1"]:.4f})', fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {PLOTS_DIR}/confusion_matrices.png")

## 8. Export Production Artifacts

In [ ]:
# Save encoder-only weights (strip projection head for production inference)
encoder_state = {k.replace('encoder.', '', 1): v
                 for k, v in model.state_dict().items() if k.startswith('encoder.')}
torch.save(encoder_state, f'{PROD_DIR}/encoder_weights.pt')

# Save production meta — matches what detection-service/detector.py expects
prod_meta = {
    'feature_dim':     FEATURE_DIM,
    'embed_dim':       EMBED_DIM,
    'threshold':       THRESHOLD,
    'dropout':         BEST_CONFIG.get('dropout', 0.3),
    'hidden_dims':     list(hidden_dims),
    'dataset_trained': DATASET_CHOICE,
    'val_auc':         meta['val_auc'],
    'test_auc':        meta['test_auc'],
    'eval_results':    {k: {m: v[m] for m in ['AUC','F1','N']}
                        for k, v in results.items()},
    'feature_names':   FEATURE_NAMES,
}
with open(f'{PROD_DIR}/model_meta.json', 'w') as f:
    json.dump(prod_meta, f, indent=2)

joblib.dump(scaler, f'{PROD_DIR}/scaler.joblib')
np.save(f'{PROD_DIR}/centroid.npy', centroid)

print("Production artifacts saved to:", PROD_DIR)
print("  encoder_weights.pt")
print("  scaler.joblib")
print("  centroid.npy")
print("  model_meta.json")
print("\nDownload from Drive and place in: detection-service/models/")

## 9. ONNX Export (Optional — Fast CPU Inference)

In [ ]:
try:
    dummy     = torch.randn(1, FEATURE_DIM).to(DEVICE)
    onnx_path = f'{PROD_DIR}/encoder.onnx'
    torch.onnx.export(
        model.encoder, dummy, onnx_path,
        input_names=['features'], output_names=['embedding'],
        dynamic_axes={'features': {0: 'batch'}, 'embedding': {0: 'batch'}},
        opset_version=14,
    )
    print(f"ONNX model saved: {onnx_path}")
    # Verify
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model verification passed.")
except Exception as e:
    print(f"ONNX export failed (non-critical): {e}")